# Wilcoxon Signed-Rank Test for p-value

In [4]:
import pandas as pd
from scipy.stats import wilcoxon

# 1. Load the Data
# NOTE: Ensure these files contain the data you want to compare.
# Currently, they contain 100 Iterations of the Best Run.
try:
    df_poa = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/docs/mlflow/all/mlflow_iteration_100_1/poa_results.csv')['fitness']
    df_pso = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/docs/mlflow/all/mlflow_iteration_100_1/pso_results.csv')['fitness']
    df_aco = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/docs/mlflow/all/mlflow_iteration_100_1/aco_results.csv')['fitness']
    df_abc = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/docs/mlflow/all/mlflow_iteration_100_1/abc_results.csv')['fitness']
except FileNotFoundError:
    print("Error: Please upload the .csv files first.")

# 2. Define a Function for the Wilcoxon Test
def perform_test(name1, data1, name2, data2):
    # 'alternative="less"' checks if data1 is LESS (better) than data2
    # We take the first N elements where N is the minimum length
    min_len = min(len(data1), len(data2))
    d1 = data1[:min_len]
    d2 = data2[:min_len]
    
    # Perform the test
    # If all differences are zero (identical), Wilcoxon isn't needed
    if all(x == y for x, y in zip(d1, d2)):
        print(f"{name1} vs {name2}: IDENTICAL (No difference)")
        return

    stat, p_value = wilcoxon(d1, d2, alternative='less')
    
    print(f"--- {name1} vs {name2} ---")
    print(f"Statistic: {stat}")
    print(f"P-Value:   {p_value:.5e}")
    if p_value < 0.05:
        print("Result:    SIGNIFICANT (Hypothesis Accepted)\n")
    else:
        print("Result:    NOT SIGNIFICANT (Hypothesis Rejected)\n")

# 3. Run the Comparisons
print("=== WILCOXON SIGNED-RANK TEST RESULTS ===\n")
perform_test('POA', df_poa, 'PSO', df_pso)
perform_test('POA', df_poa, 'ACO', df_aco)
perform_test('POA', df_poa, 'ABC', df_abc)

=== WILCOXON SIGNED-RANK TEST RESULTS ===

--- POA vs PSO ---
Statistic: 5.0
P-Value:   2.26288e-18
Result:    SIGNIFICANT (Hypothesis Accepted)

--- POA vs ACO ---
Statistic: 4.0
P-Value:   2.19559e-18
Result:    SIGNIFICANT (Hypothesis Accepted)

--- POA vs ABC ---
Statistic: 4.0
P-Value:   2.19436e-18
Result:    SIGNIFICANT (Hypothesis Accepted)



## Saving into .csv format

In [48]:
import pandas as pd
from scipy.stats import wilcoxon
import os

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
# Define your file paths here
base_path = '/home/emery/Documents/Scheduling_Puma_Optimizer/docs/mlflow/all/mlflow_iteration_100_20/'
files = {
    'POA': base_path + 'poa_results.csv',
    'PSO': base_path + 'pso_results.csv',
    'ACO': base_path + 'aco_results.csv',
    'ABC': base_path + 'abc_results.csv'
}

dataframes = {}
print("--- Loading Data ---")
for algo, filepath in files.items():
    try:
        if os.path.exists(filepath):
            # We only need the 'fitness' column
            df = pd.read_csv(filepath)
            if 'fitness' in df.columns:
                dataframes[algo] = df['fitness']
                print(f"Loaded {algo}: {len(df)} rows")
            else:
                print(f"Error: 'fitness' column missing in {filepath}")
        else:
            print(f"Error: File not found at {filepath}")
    except Exception as e:
        print(f"Error reading {algo}: {e}")
print("")

# ==========================================
# 2. DEFINE TEST FUNCTION
# ==========================================
def perform_wilcoxon_test(name1, data1, name2, data2):
    """
    Performs Wilcoxon Signed-Rank Test and returns a dictionary of results.
    """
    # Ensure we compare the same number of data points
    min_len = min(len(data1), len(data2))
    d1 = data1[:min_len]
    d2 = data2[:min_len]
    
    # Initialize result dictionary
    result = {
        'Comparison': f"{name1} vs {name2}",
        'Algorithm_1': name1,
        'Algorithm_2': name2,
        'N_Samples': min_len,
        'Statistic': 0.0,
        'P_Value': 1.0,
        'Significance': 'Not Significant'
    }

    # Check if arrays are identical (Wilcoxon fails on identical data)
    if all(x == y for x, y in zip(d1, d2)):
        result['Significance'] = "IDENTICAL (No Difference)"
        return result

    # Perform the test (alternative='less' checks if d1 is LESS/BETTER than d2)
    try:
        stat, p_value = wilcoxon(d1, d2, alternative='less')
        result['Statistic'] = stat
        result['P_Value'] = p_value
        
        if p_value < 0.05:
            result['Significance'] = "SIGNIFICANT (p < 0.05)"
        else:
            result['Significance'] = "NOT SIGNIFICANT (p >= 0.05)"
            
    except Exception as e:
        result['Significance'] = f"Error: {str(e)}"
        
    return result

# ==========================================
# 3. RUN COMPARISONS & SAVE TO CSV
# ==========================================
results_list = []

if 'POA' in dataframes:
    print("=== WILCOXON SIGNED-RANK TEST RESULTS ===\n")
    targets = ['PSO', 'ACO', 'ABC']
    
    for target in targets:
        if target in dataframes:
            # Run the test
            res = perform_wilcoxon_test('POA', dataframes['POA'], target, dataframes[target])
            
            # Print to console for immediate feedback
            print(f"--- {res['Comparison']} ---")
            print(f"Statistic: {res['Statistic']}")
            print(f"P-Value:   {res['P_Value']:.5e}")
            print(f"Result:    {res['Significance']}\n")
            
            # Add to list
            results_list.append(res)

# Convert list of results to a DataFrame
if results_list:
    df_results = pd.DataFrame(results_list)
    
    # Define output filename
    output_csv = 'wilcoxon_test_results.csv'
    
    # Save to CSV
    df_results.to_csv(output_csv, index=False)
    print(f"✅ Success! All statistical results saved to: {output_csv}")
    print(df_results)
else:
    print("❌ No comparisons were performed. Check your data loading.")

--- Loading Data ---
Loaded POA: 100 rows
Loaded PSO: 100 rows
Loaded ACO: 100 rows
Loaded ABC: 100 rows

=== WILCOXON SIGNED-RANK TEST RESULTS ===

--- POA vs PSO ---
Statistic: 0.0
P-Value:   1.94542e-18
Result:    SIGNIFICANT (p < 0.05)

--- POA vs ACO ---
Statistic: 0.0
P-Value:   1.94553e-18
Result:    SIGNIFICANT (p < 0.05)

--- POA vs ABC ---
Statistic: 0.0
P-Value:   1.94202e-18
Result:    SIGNIFICANT (p < 0.05)

✅ Success! All statistical results saved to: wilcoxon_test_results.csv
   Comparison Algorithm_1 Algorithm_2  N_Samples  Statistic       P_Value  \
0  POA vs PSO         POA         PSO        100        0.0  1.945424e-18   
1  POA vs ACO         POA         ACO        100        0.0  1.945534e-18   
2  POA vs ABC         POA         ABC        100        0.0  1.942025e-18   

             Significance  
0  SIGNIFICANT (p < 0.05)  
1  SIGNIFICANT (p < 0.05)  
2  SIGNIFICANT (p < 0.05)  


# Friedman Test (Global Test)

In [49]:
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
import itertools
import os
import numpy as np

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
# Update this path to your actual folder
base_path = '/home/emery/Documents/Scheduling_Puma_Optimizer/docs/mlflow/all/mlflow_iteration_100_20/'

files = {
    'POA': base_path + 'poa_results.csv',
    'PSO': base_path + 'pso_results.csv',
    'ACO': base_path + 'aco_results.csv',
    'ABC': base_path + 'abc_results.csv'
}

data = {}
print("--- Loading Data ---")
try:
    for name, filepath in files.items():
        if os.path.exists(filepath):
            # Reading the 'fitness' column
            df = pd.read_csv(filepath)
            data[name] = df['fitness'].values
            print(f"Loaded {name}: {len(data[name])} samples")
        else:
            print(f"Warning: {name} file not found at {filepath}")
except Exception as e:
    print(f"Error loading data: {e}")

# Ensure we have data to compare
if len(data) < 2:
    print("Error: Need at least 2 algorithms to compare.")
    exit()

# Align data lengths (Truncate to minimum length found)
min_len = min(len(v) for v in data.values())
algo_names = list(data.keys())
dataset_matrix = [data[name][:min_len] for name in algo_names]

# List to collect all results for CSV
results_csv = []

# ==========================================
# 2. GLOBAL COMPARISON (FRIEDMAN TEST)
# ==========================================
print("\n=== 1. FRIEDMAN TEST (Global Comparison) ===")
stat, p_global = friedmanchisquare(*dataset_matrix)

friedman_result = "SIGNIFICANT (Differences Detected)" if p_global < 0.05 else "NOT SIGNIFICANT"

print(f"Statistic: {stat:.4f}")
print(f"P-Value:   {p_global:.5e}")
print(f"Result:    {friedman_result}")

results_csv.append({
    'Test_Type': 'Global (Friedman)',
    'Comparison': 'All Algorithms',
    'Statistic': stat,
    'P_Value': p_global,
    'Significance': friedman_result,
    'Alpha_Threshold': 0.05
})

# ==========================================
# 3. POST-HOC ANALYSIS (PAIRWISE WILCOXON)
# ==========================================
# Only run if Friedman is significant
if p_global < 0.05:
    print("\n=== 2. POST-HOC PAIRWISE COMPARISONS (Bonferroni Corrected) ===")
    
    # Calculate Bonferroni Correction
    # Formula: Alpha / Number of Pairs
    # For 4 Algos, pairs = 6. So 0.05 / 6 = 0.0083
    n_algos = len(algo_names)
    n_pairs = (n_algos * (n_algos - 1)) / 2
    alpha_corrected = 0.05 / n_pairs
    
    print(f"Corrected Alpha Threshold: {alpha_corrected:.6f} (0.05 / {int(n_pairs)} pairs)")
    
    # Generate all pairs (POA vs PSO, POA vs ACO, etc.)
    pairs = list(itertools.combinations(algo_names, 2))
    
    for name1, name2 in pairs:
        d1 = data[name1][:min_len]
        d2 = data[name2][:min_len]
        
        # Check for identical data
        if np.array_equal(d1, d2):
            res_text = "IDENTICAL"
            s, p = 0.0, 1.0
        else:
            # Two-sided Wilcoxon to see if they are different
            s, p = wilcoxon(d1, d2)
            
            if p < alpha_corrected:
                res_text = "SIGNIFICANT"
            else:
                res_text = "NOT SIGNIFICANT"
        
        print(f"{name1} vs {name2}: p={p:.5e} -> {res_text}")
        
        results_csv.append({
            'Test_Type': 'Post-hoc (Wilcoxon)',
            'Comparison': f"{name1} vs {name2}",
            'Statistic': s,
            'P_Value': p,
            'Significance': res_text,
            'Alpha_Threshold': alpha_corrected
        })

# ==========================================
# 4. SAVE RESULTS TO CSV
# ==========================================
output_filename = 'friedman_test_results.csv'
df_results = pd.DataFrame(results_csv)
df_results.to_csv(output_filename, index=False)

print(f"\n✅ All results saved to: {output_filename}")
print(df_results)

--- Loading Data ---
Loaded POA: 100 samples
Loaded PSO: 100 samples
Loaded ACO: 100 samples
Loaded ABC: 100 samples

=== 1. FRIEDMAN TEST (Global Comparison) ===
Statistic: 300.0000
P-Value:   9.94876e-65
Result:    SIGNIFICANT (Differences Detected)

=== 2. POST-HOC PAIRWISE COMPARISONS (Bonferroni Corrected) ===
Corrected Alpha Threshold: 0.008333 (0.05 / 6 pairs)
POA vs PSO: p=3.89085e-18 -> SIGNIFICANT
POA vs ACO: p=3.89107e-18 -> SIGNIFICANT
POA vs ABC: p=3.88405e-18 -> SIGNIFICANT
PSO vs ACO: p=2.41524e-21 -> SIGNIFICANT
PSO vs ABC: p=3.82228e-18 -> SIGNIFICANT
ACO vs ABC: p=3.83482e-18 -> SIGNIFICANT

✅ All results saved to: friedman_test_results.csv
             Test_Type      Comparison  Statistic       P_Value  \
0    Global (Friedman)  All Algorithms      300.0  9.948758e-65   
1  Post-hoc (Wilcoxon)      POA vs PSO        0.0  3.890849e-18   
2  Post-hoc (Wilcoxon)      POA vs ACO        0.0  3.891068e-18   
3  Post-hoc (Wilcoxon)      POA vs ABC        0.0  3.884050e-18  